# Evaluierung & Analyse: Temperature Ladder DPO Experiment (500 Tokens)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook analysiert und vergleicht das **500-Token mBART-50 Modell** vor und nach dem **Direct Preference Optimization (DPO)** Training mit dem **Progressive Temperature Ladder Verfahren** auf dem ungesehenen **10kGNAD Alltagssprache-Korpus**.

### Untersuchte Modellkonfigurationen:
1. **SFT Baseline (500 Tokens):** Supervised Fine-Tuning Referenzmodell (`results/models/token_length_exp/sft_len500`)
2. **DPO Ladder (500 Tokens):** DPO Modell trainiert mit Temperature Ladder Präferenzpaaren (`results/models/temperature_ladder_500/dpo_w05_w05`)

### Kernfragen der Analyse:
- **Erhöht das Temperature Ladder DPO Training die Einfachheit ($R_{\text{style}}$), ohne Fakten zu verlieren ($R_{\text{sem}}$)?**
- **Wurden Wort- und Phrasenwiederholungen durch die Anti-Repetition-Constraints (`ngram=3`, `rep_penalty=1.35`) eliminiert?**
- **Wie verhält sich die Kompressionsrate und Satzstruktur im Vergleich zur SFT-Baseline?**


In [ ]:
# ==============================================================================
# 1. SYSTEM-SETUP & IMPORTE
# ==============================================================================
import os
import sys
import json
import glob
import random
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import ttest_rel, wilcoxon

import torch
import torch.nn as nn
import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer, util

# Arbeitsverzeichnis robust auf Projekt-Root setzen
while not (os.path.exists(".git") or os.path.exists("data")) and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

for candidate in [
    "/home/fiete/master-thesis",
    "/home/fisc4884/master-thesis",
    "/Users/fietescheel/Documents/Master Thesis",
    os.path.expanduser("~/Documents/Master Thesis"),
    os.path.expanduser("~/master-thesis")
]:
    if os.path.exists(candidate):
        os.chdir(candidate)
        break

print("Arbeitsverzeichnis:", os.getcwd())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Nutze Device: {DEVICE}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 150


In [ ]:
# ==============================================================================
# 2. REWARD-EVALUATOR & METRIK-DEFINITIONEN (500 TOKENS)
# ==============================================================================
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 128, dropout: float = 0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out)

# Lade SpaCy Tokenizer
try:
    nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])
except Exception:
    nlp = spacy.blank("de")
    nlp.add_pipe("sentencizer")

# Lade 500er BiLSTM Regressor
REWARD_MODEL_PATH = "results/models/token_length_exp/bilstm_mixup_regression_500.pt"
REWARD_VOCAB_PATH = "data/token_length_exp/mixup_vocab_500.json"

with open(REWARD_VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)
    stoi = vocab_data.get("stoi", vocab_data)
unk_idx = stoi.get("<unk>") or stoi.get("<UNK>") or 1

bilstm_model = BiLSTMRegressor(vocab_size=len(stoi), embed_dim=128, hidden_dim=128).to(DEVICE)
raw_state = torch.load(REWARD_MODEL_PATH, map_location=DEVICE)
if isinstance(raw_state, dict) and "model_state_dict" in raw_state:
    raw_state = raw_state["model_state_dict"]
bilstm_model.load_state_dict(raw_state)
bilstm_model.eval()

# Lade SBERT
sbert_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device=DEVICE)
print("Evaluatoren erfolgreich geladen.")


In [ ]:
# ==============================================================================
# 3. EVALUATION HELPER & METRIK-BERECHNUNG
# ==============================================================================
def calculate_trigram_diversity(text: str) -> float:
    words = text.lower().split()
    if len(words) < 3:
        return 1.0
    trigrams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    return len(set(trigrams)) / len(trigrams)

def evaluate_model_on_benchmark(model_path: str, base_model_name: str, test_samples: List[Dict], batch_size: int = 4):
    print(f"\nLade Modell: {model_path}...")
    dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        
    if hasattr(tokenizer, "src_lang") or hasattr(tokenizer, "lang_code_to_id"):
        tokenizer.src_lang = "de_DE"
        tokenizer.tgt_lang = "de_DE"
        
    has_adapter = os.path.exists(os.path.join(model_path, "adapter_config.json"))
    if has_adapter:
        base_m = AutoModelForSeq2SeqLM.from_pretrained(base_model_name, torch_dtype=dtype)
        peft_m = PeftModel.from_pretrained(base_m, model_path)
        model = peft_m.merge_and_unload()
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=dtype)
        
    model = model.to(DEVICE)
    model.eval()
    
    as_texts = [str(s.get("as_text") or "").strip() for s in test_samples]
    ref_texts = [str(s.get("ls_text") or "").strip() for s in test_samples]
    
    translations = []
    num_batches = (len(as_texts) + batch_size - 1) // batch_size
    for b in tqdm(range(num_batches), desc=f"Generiere {os.path.basename(model_path)}"):
        prompts = as_texts[b * batch_size : (b + 1) * batch_size]
        inp = tokenizer(prompts, padding=True, truncation=True, max_length=500, return_tensors="pt").to(DEVICE)
        
        gen_kwargs = {
            "input_ids": inp["input_ids"],
            "attention_mask": inp.get("attention_mask"),
            "max_length": 500,
            "repetition_penalty": 1.35,
            "no_repeat_ngram_size": 3,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
        }
        if hasattr(tokenizer, "lang_code_to_id") and "de_DE" in tokenizer.lang_code_to_id:
            gen_kwargs["forced_bos_token_id"] = tokenizer.lang_code_to_id["de_DE"]
            
        with torch.no_grad():
            out = model.generate(**gen_kwargs)
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        translations.extend([d.strip() for d in decoded])
        
    # Scores berechnen
    style_scores = []
    for text in translations:
        doc = nlp(str(text or ""))
        tokens = [t.text.lower() for t in doc if not t.is_space]
        indices = [stoi.get(t, unk_idx) for t in tokens[:500]]
        if len(indices) == 0:
            indices = [0]
        inp_tensor = torch.tensor([indices], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            style_scores.append(bilstm_model(inp_tensor).item())
    style_scores = np.array(style_scores)
    
    with torch.inference_mode():
        emb_as = sbert_model.encode(as_texts, batch_size=8, convert_to_tensor=True, show_progress_bar=False)
        emb_cand = sbert_model.encode(translations, batch_size=8, convert_to_tensor=True, show_progress_bar=False)
        emb_ref = sbert_model.encode(ref_texts, batch_size=8, convert_to_tensor=True, show_progress_bar=False)
        
        sim_as = util.cos_sim(emb_as, emb_cand).diagonal().cpu().numpy()
        sim_ref = util.cos_sim(emb_ref, emb_cand).diagonal().cpu().numpy()
        
    sim_as_norm = np.clip((sim_as + 1.0) / 2.0, 0.0, 1.0)
    composite_rewards = 0.5 * style_scores + 0.5 * sim_as_norm
    diversities = [calculate_trigram_diversity(t) for t in translations]
    comp_ratios = [len(t.split()) / max(1, len(a.split())) for t, a in zip(translations, as_texts)]
    
    df_res = pd.DataFrame({
        "as_text": as_texts,
        "ls_reference": ref_texts,
        "translation": translations,
        "style_score": style_scores,
        "sbert_to_as": sim_as,
        "sbert_to_ref": sim_ref,
        "composite_reward": composite_rewards,
        "trigram_diversity": diversities,
        "compression_ratio": comp_ratios,
    })
    return df_res


In [ ]:
# ==============================================================================
# 4. BENCHMARK-EVALUATION AUSFÜHREN (LEBENSHILFE TESTSET)
# ==============================================================================
TEST_DATA_PATH = "data/lebenshilfe/lebenshilfe_dataset_clean.json"
with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
    test_samples = json.load(f)

models = {
    "SFT Baseline (500 Tokens)": "results/models/token_length_exp/sft_len500",
    "DPO Ladder (500 Tokens)": "results/models/temperature_ladder_500/dpo_w05_w05",
}

results = {}
for name, path in models.items():
    if os.path.exists(path):
        results[name] = evaluate_model_on_benchmark(path, "facebook/mbart-large-50", test_samples)
    else:
        print(f"Modellordner nicht gefunden (noch im Training?): {path}")


In [ ]:
# ==============================================================================
# 5. METRIKEN-VERGLEICHSTABELLE
# ==============================================================================
if len(results) > 0:
    summary_rows = []
    for name, df in results.items():
        summary_rows.append({
            "Modell": name,
            "Ø Style-Score (R_style)": f"{df['style_score'].mean():.4f} ± {df['style_score'].std():.4f}",
            "Ø SBERT zu AS (R_sem)": f"{df['sbert_to_as'].mean():.4f} ± {df['sbert_to_as'].std():.4f}",
            "Ø SBERT zu Gold LS": f"{df['sbert_to_ref'].mean():.4f} ± {df['sbert_to_ref'].std():.4f}",
            "Ø Composite Reward": f"{df['composite_reward'].mean():.4f} ± {df['composite_reward'].std():.4f}",
            "Ø Trigram-Diversität": f"{df['trigram_diversity'].mean()*100:.1f} %",
            "Ø Kompressionsrate": f"{df['compression_ratio'].mean():.3f}",
        })
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)


In [ ]:
# ==============================================================================
# 6. STATISTISCHE SIGNIFIKANZTESTS (PAIRED T-TEST & WILCOXON)
# ==============================================================================
if "SFT Baseline (500 Tokens)" in results and "DPO Ladder (500 Tokens)" in results:
    sft_df = results["SFT Baseline (500 Tokens)"]
    dpo_df = results["DPO Ladder (500 Tokens)"]
    
    metrics = ["style_score", "sbert_to_as", "sbert_to_ref", "composite_reward", "trigram_diversity"]
    sig_rows = []
    
    for m in metrics:
        t_stat, p_val_t = ttest_rel(dpo_df[m], sft_df[m])
        w_stat, p_val_w = wilcoxon(dpo_df[m], sft_df[m])
        delta = dpo_df[m].mean() - sft_df[m].mean()
        sig_rows.append({
            "Metrik": m,
            "SFT Mittelwert": f"{sft_df[m].mean():.4f}",
            "DPO Mittelwert": f"{dpo_df[m].mean():.4f}",
            "Δ Differenz": f"{delta:+.4f}",
            "p-Wert (Paired t-Test)": f"{p_val_t:.4e}",
            "Signifikant (p < 0.05)": "✅ Ja" if p_val_t < 0.05 else "❌ Nein",
        })
        
    df_sig = pd.DataFrame(sig_rows)
    display(df_sig)


In [ ]:
# ==============================================================================
# 7. VISUALISIERUNGEN (KDE-VERTEILUNGEN & PARETO-SCATTERPLOT)
# ==============================================================================
if len(results) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Style Score KDE
    for name, df in results.items():
        sns.kdeplot(df["style_score"], ax=axes[0], label=name, fill=True, alpha=0.3)
    axes[0].set_title("Einfachheits-Score ($R_{style}$)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Score")
    axes[0].legend()
    
    # 2. SBERT Semantik KDE
    for name, df in results.items():
        sns.kdeplot(df["sbert_to_as"], ax=axes[1], label=name, fill=True, alpha=0.3)
    axes[1].set_title("Semantischer Erhalt zu AS ($R_{sem}$)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("SBERT Cosine Similarity")
    axes[1].legend()
    
    # 3. Composite Reward KDE
    for name, df in results.items():
        sns.kdeplot(df["composite_reward"], ax=axes[2], label=name, fill=True, alpha=0.3)
    axes[2].set_title("Composite Reward ($0.5 \cdot R_{style} + 0.5 \cdot R_{sem}$)", fontsize=13, fontweight="bold")
    axes[2].set_xlabel("Score")
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Pareto Plot
    plt.figure(figsize=(9, 6))
    for name, df in results.items():
        plt.scatter(df["style_score"], df["sbert_to_as"], label=name, alpha=0.6, s=40)
        plt.scatter(df["style_score"].mean(), df["sbert_to_as"].mean(), marker="X", s=200, edgecolors="black", label=f"Ø {name}")
        
    plt.title("Pareto-Vergleich: Simplicity ($R_{style}$) vs. Bedeutungserhalt ($R_{sem}$)", fontsize=14, fontweight="bold")
    plt.xlabel("Style Simplicity Score ($R_{style}$)", fontsize=12)
    plt.ylabel("Semantic Preservation to AS ($R_{sem}$)", fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# 8. QUALITATIVE STICHPROBEN IM DIREKTEN VERGLEICH
# ==============================================================================
if "SFT Baseline (500 Tokens)" in results and "DPO Ladder (500 Tokens)" in results:
    sft_df = results["SFT Baseline (500 Tokens)"]
    dpo_df = results["DPO Ladder (500 Tokens)"]
    
    for i in range(min(5, len(sft_df))):
        print(f"\n{'='*90}")
        print(f"[STICHPROBE {i+1}]")
        print(f"{'='*90}")
        print(f"AS-ORIGINAL:   {sft_df.iloc[i]['as_text'][:220]}...")
        print(f"GOLD LS-REF:   {sft_df.iloc[i]['ls_reference'][:220]}...")
        print(f"{'-'*90}")
        print(f"SFT-OUTPUT:    {sft_df.iloc[i]['translation']}")
        print(f"Scores (SFT):  Style={sft_df.iloc[i]['style_score']:.3f} | Sem={sft_df.iloc[i]['sbert_to_as']:.3f} | Trigram-Div={sft_df.iloc[i]['trigram_diversity']*100:.1f}%")
        print(f"{'-'*90}")
        print(f"DPO-LADDER:    {dpo_df.iloc[i]['translation']}")
        print(f"Scores (DPO):  Style={dpo_df.iloc[i]['style_score']:.3f} | Sem={dpo_df.iloc[i]['sbert_to_as']:.3f} | Trigram-Div={dpo_df.iloc[i]['trigram_diversity']*100:.1f}%")
        print(f"{'='*90}\n")
